In [1]:
import torch

In [2]:
def naive_attention(Q,K,V):

    d_k = Q.shape[-1]

    score = Q @ K.transpose(-2,-1) / d_k**0.5
    attn = torch.softmax(score, dim=-1)
    out = attn @ V
    return out


In [3]:
B, H, T, d_k = 1, 1, 16, 32
Q = torch.randn(B, H, T, d_k)
K = torch.randn(B, H, T, d_k)
V = torch.randn(B, H, T, d_k)

out = naive_attention(Q, K, V)
print(out.shape)  # should be (1, 1, 16, 32)

torch.Size([1, 1, 16, 32])


In [4]:
Q_block = Q[:, :, 0:16, :]  # first block
K_block = K[:, :, 0:16, :]
V_block = V[:, :, 0:16, :]

score = Q_block @ K_block.transpose(-2,-1) / d_k**0.5
attn = torch.softmax(score, dim=-1)
out_block = attn @ V_block

In [9]:
def flash_attention_forward(Q, K, V, tile_size=16):

    B, H ,T, d_k = Q.shape
    out = torch.zeros(B, H, T, d_k)

    for i in range(0, T, tile_size):

        Q_block = Q[:, :, i:i+tile_size, :]
        m = torch.full((B, H, tile_size), float('-inf'))
        run_denom = torch.zeros(B, H, tile_size)
        run_numer = torch.zeros(B, H, tile_size, d_k)

        for j in range(0, T, tile_size):

            K_block = K[:, :, j:j+tile_size, :]
            V_block = V[:, :, j:j+tile_size, :]

            score = Q_block @ K_block.transpose(-2,-1) / d_k**0.5

            m_new = torch.maximum(m, torch.max(score, dim=-1).values)
            corr_factor = torch.exp(m - m_new)
            run_denom = run_denom * corr_factor + torch.exp(score - m_new.unsqueeze(-1)).sum(dim=-1)
            run_numer = run_numer * corr_factor.unsqueeze(-1) + torch.exp(score - m_new.unsqueeze(-1)) @ V_block
            m = m_new

        out_block = run_numer / run_denom.unsqueeze(-1)
        out[:, :, i:i+tile_size, :] = out_block

    return out


In [10]:
B, H, T, d_k = 1, 1, 64, 32

Q = torch.randn(B, H, T, d_k)
K = torch.randn(B, H, T, d_k)
V = torch.randn(B, H, T, d_k)

naive_out = naive_attention(Q, K, V)
flash_out = flash_attention_forward(Q, K, V, tile_size=16)

print(torch.allclose(naive_out, flash_out, atol=1e-5))

True


In [11]:
import time

for T in [128, 256, 512, 1024, 2048]:
    Q = torch.randn(1, 1, T, 64)
    K = torch.randn(1, 1, T, 64)
    V = torch.randn(1, 1, T, 64)
    
    start = time.time()
    naive_attention(Q, K, V)
    naive_time = time.time() - start
    
    start = time.time()
    flash_attention_forward(Q, K, V, tile_size=16)
    flash_time = time.time() - start
    
    print(f"T={T} | Naive: {naive_time*1000:.2f}ms | Flash: {flash_time*1000:.2f}ms")

T=128 | Naive: 15.75ms | Flash: 38.57ms
T=256 | Naive: 3.00ms | Flash: 121.76ms
T=512 | Naive: 8.61ms | Flash: 349.16ms
T=1024 | Naive: 10.47ms | Flash: 1255.11ms
T=2048 | Naive: 28.50ms | Flash: 5697.83ms
